# Practical on HYperrealistic neural coding of PERception (HYPER)

In this practical, you will get hands-on experience with neural encoding and neural decoding via a GAN latent space. You will be working in the same groups as the ones you did the presentations and the deadline is: May 30th, Friday. You can get a total of 80 points which translates to 20% (or 2/10) of your total grade for the course.

### Reminder! The workgroups for this assignment, where you can ask questions to the TAs are: 16th May -Friday-, 23rd May -Friday- and 26th May -Monday!!!-. The practical on 26th May will happen in MM 2.130, the same classroom as the lectures take place.

The practical consists of the following parts:

1.   PGGAN
2.   Dataset
3.   Neural encoding
4.   Neural decoding
5.   Your own experiment

We provide you with the MXNET implementation of NVIDIA's progressively grown generative adversarial network (PGGAN) generator network for face generation that was used in [the paper](https://www.nature.com/articles/s41598-021-03938-w).

In the first and second part, you will familiarize yourselves with PGGAN and the (stimulus, response) dataset. In the third and fourth part, you will implement neural encoding (face2brain) and neural decoding (brain2face) pipelines. In the last part, you will conduct your own experiment that you will write an "ultra-short" report on.

⚠️ Beware! You cannot edit this document. Go to 'File/Save a copy in Drive' to work on a version specific to you which you can edit to your heart's content.

## Note for the initial steps
The following code is a necessary step for some of the libraries required for this assignment to work. Unfortunately, you will have to run these blocks every time you disconnect from the session, meaning that you will need approximately 10 minutes. We apologize for this inconvinence.

The blocks below are for housekeeping and they install, download, load, define, and do a bit more. That being said, understanding them is not the focus of this assignment, so also feel free to just run them and be done with it.

Make sure that your google colab runtime is using a GPU (Runtime → Change runtime type).

In [ ]:
# STEP 1: Remove existing CUDA and cuDNN
!apt-get purge -y "*cublas*" "*cufft*" "*curand*" "*cusolver*" "*cusparse*" "*npp*" "*nvjpeg*" "cuda*" "nsight*"
!rm -rf /usr/local/cuda*

# STEP 2: Install CUDA 11.2
!wget https://developer.download.nvidia.com/compute/cuda/11.2.2/local_installers/cuda_11.2.2_460.32.03_linux.run
!chmod +x cuda_11.2.2_460.32.03_linux.run
!./cuda_11.2.2_460.32.03_linux.run --silent --toolkit --override

# STEP 3: Install cuDNN 8.1 for CUDA 11.2
!wget https://developer.download.nvidia.com/compute/redist/cudnn/v8.1.1/cudnn-11.2-linux-x64-v8.1.1.33.tgz
!tar -xzvf cudnn-11.2-linux-x64-v8.1.1.33.tgz
!cp cuda/include/cudnn*.h /usr/local/cuda/include/
!cp cuda/lib64/libcudnn* /usr/local/cuda/lib64/
!chmod a+r /usr/local/cuda/include/cudnn*.h /usr/local/cuda/lib64/libcudnn*

# STEP 4: Add CUDA to dynamic linker config
!echo "/usr/local/cuda/lib64" > /etc/ld.so.conf.d/cuda.conf
!ldconfig

# STEP 5: Install compatible NumPy
!pip install numpy==1.23.5 --force-reinstall

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'cublasmp-cuda-11' for glob '*cublas*'
Note, selecting 'cublasmp-cuda-12' for glob '*cublas*'
Note, selecting 'cublasmp0' for glob '*cublas*'
Note, selecting 'libcublasmp0-dev-cuda-11' for glob '*cublas*'
Note, selecting 'libcublasmp0-dev-cuda-12' for glob '*cublas*'
Note, selecting 'libcublasmp0-cuda-11' for glob '*cublas*'
Note, selecting 'libcublasmp0-cuda-12' for glob '*cublas*'
Note, selecting 'libcublas.so.11' for glob '*cublas*'
Note, selecting 'libcublas.so.12' for glob '*cublas*'
Note, selecting 'libcublas-dev-11-0' for glob '*cublas*'
Note, selecting 'libcublas-dev-11-1' for glob '*cublas*'
Note, selecting 'libcublas-dev-11-7' for glob '*cublas*'
Note, selecting 'libcublas-dev-11-8' for glob '*cublas*'
Note, selecting 'libcublas-dev-12-0' for glob '*cublas*'
Note, selecting 'libcublas-dev-12-1' for glob '*cublas*'
Note, selecting 'libcublas-dev-12-2' for glob '*cub

In [ ]:
!python --version

# STEP 6: Now restart the session

> Runtime > Restart Session

This clears all Python modules, ensuring a clean environment where numpy==1.23.5 is respected

In [ ]:
# STEP 7: Set environment variables
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.2'
os.environ['PATH'] = '/usr/local/cuda-11.2/bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.2/lib64'

# STEP 8: Install MXNet and other libraries
!pip install mxnet-cu112
!pip install nibabel nilearn tqdm

# STEP 9: Test MXNet GPU support
import numpy as np
np.bool = np.bool_ # needs to be here for mxnet to import. don't question it
import mxnet as mx
print("MXNet version:", mx.__version__)
#print("GPUs available:", mx.test_utils.list_gpus())

The block below is defining where all of the data and files are being stored that you'll generate and download within this notebook. Per default, it stores everything in the temporary runtime of google colab, which means all files are being deleted when the runtime ends. This is enough to run it, create all the output and then (later) download it to submit it.
However, if you have downloaded this notebook to run it locally, or wish to save files to your google drive folder for safekeeping, you will need to make changes to the block as outlined in the comments.

In [ ]:
from pathlib import Path

# all downloaded files will be stored here. This works great in the colab environment
# although all of your files will be deleted when you end the colab session
# if you want to run this code locally, you might want to change this to a more suited path
# e.g. "." to save in the current directory
__working_directory = Path("/content/")

# if you want to save the files you are generating to your google drive when running in colab
# you can uncomment these lines of code
# from google.colab import drive
# drive.mount('/content/drive')
# __working_directory = __working_directory / "drive" / "ai4nt_2425"
# __working_directory.mkdir(parents=True, exist_ok=True)

__save_location = __working_directory / "downloads/"
__save_location.mkdir(parents=True, exist_ok=True)

In [ ]:
from __future__ import annotations

# import os
import hashlib
import io
import warnings
import pickle
import re
import requests
from types import ModuleType
from typing import Tuple, Union

from tqdm.notebook import trange, tqdm
import matplotlib.pyplot as plt
import matplotlib.image as img
from matplotlib import cm
import numpy as np
# import mxnet as mx
from mxnet import autograd, Context, gluon, nd, symbol
from mxnet.gluon import loss, nn
from mxnet.gluon.nn import Conv2D, Dense, HybridBlock, HybridSequential, LeakyReLU
from mxnet.gluon.parameter import Parameter
from mxnet.initializer import Zero
from mxnet.ndarray import NDArray
from mxnet.io import NDArrayIter
import nibabel as nib
from nilearn import plotting
from PIL import Image
from scipy.stats import t, zscore
from sklearn.linear_model import RidgeCV

In [ ]:
# helper functions to download and load all the data files spread around the internet

def _url_to_name(url:str) -> str:
  """Calculates the md5 hash of an url string as a proxy filename"""
  return hashlib.md5(url.encode()).hexdigest()

def get_drive_file_id_from_share_link(share_url:str) -> str:
    """From a google drive public sharing link, get the file_id we can use to directly download the file"""
    return re.match(r".*/d/(?P<file_id>[a-zA-Z0-9-_]+)/?.*", share_url)['file_id']

def make_drive_downloadable_url(file_id:str) -> str:
    """from a google drive file-id, generate a direct-download link to be used in code"""
    return r"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=y".format(file_id=file_id)

def load_from_url(url:str, name:str=None) -> bytes:
  """Loads data from a URL and returns the byte response. Implements simple caching"""
  if name is None:
    warnings.warn("Consider giving the downloaded file a unique name, to avoid accidental overwriting.", )
    name = _url_to_name(url)
  file_path = __save_location / name

  if file_path.is_file():
    with open(file_path, "rb") as f:
      res = f.read()
  else:
    with requests.get(url) as response:
      if response.status_code != 200:
        raise ConnectionError(f"Unsuccessful GET request for: {url}")
      with open(file_path, "wb") as f:
        res = response.content
        f.write(res)

  return res

def get_downloaded_file_path(name:str=None, url:str=None) -> str:
  """Some libraries deal better with file names than byte streams. Adds access to downloaded files"""
  if name is None:
    if url is None:
      raise ValueError("Name and URL cannot both be None")

    name = _url_to_name(url)

  file_path = __save_location / name
  if not file_path.is_file():
    raise FileNotFoundError(f"No file found at {file_path}\nWas it downloaded yet?")

  return str(file_path)

In [ ]:
if mx.gpu:
    ctx = mx.gpu()
else:
    ctx = mx.cpu()

print(ctx)

In [ ]:
class Pixelnorm(HybridBlock):
    def __init__(self, epsilon: float = 1e-8) -> None:
        super(Pixelnorm, self).__init__()
        self.eps = epsilon

    def hybrid_forward(self, F, x) -> nd:
        return x * F.rsqrt(F.mean(F.square(x), 1, True) + self.eps)


class Bias(HybridBlock):
    def __init__(self, shape: Tuple) -> None:
        super(Bias, self).__init__()
        self.shape = shape
        with self.name_scope():
            self.b = self.params.get("b", init=Zero(), shape=shape)

    def hybrid_forward(self, F, x, b) -> nd:
        return F.broadcast_add(x, b[None, :, None, None])


class Block(HybridSequential):
    def __init__(self, channels: int, in_channels: int) -> None:
        super(Block, self).__init__()
        self.channels = channels
        self.in_channels = in_channels
        with self.name_scope():
            self.add(Conv2D(channels, 3, padding=1, in_channels=in_channels))
            self.add(LeakyReLU(0.2))
            self.add(Pixelnorm())
            self.add(Conv2D(channels, 3, padding=1, in_channels=channels))
            self.add(LeakyReLU(0.2))
            self.add(Pixelnorm())

    def hybrid_forward(self, F, x) -> nd:
        x = F.repeat(x, 2, 2)
        x = F.repeat(x, 2, 3)
        for i in range(len(self)):
            x = self[i](x)
        return x


class Generator(HybridSequential):
    def __init__(self) -> None:
        super(Generator, self).__init__()
        with self.name_scope():
            self.add(Pixelnorm())
            self.add(Dense(8192, use_bias=False, in_units=512))
            self.add(Bias((512,)))
            self.add(LeakyReLU(0.2))
            self.add(Pixelnorm())
            self.add(Conv2D(512, 3, padding=1, in_channels=512))
            self.add(LeakyReLU(0.2))
            self.add(Pixelnorm())
            self.add(Block(512, 512))
            self.add(Block(512, 512))
            self.add(Block(512, 512))
            self.add(Block(256, 512))
            self.add(Block(128, 256))
            self.add(Block(64, 128))
            self.add(Block(32, 64))
            self.add(Block(16, 32))
            self.add(Conv2D(3, 1, in_channels=16))

    def hybrid_forward(self, F: Union(nd, symbol), x: nd, layer: int) -> nd:
        x = F.Reshape(self[1](self[0](x)), (-1, 512, 4, 4))
        for i in range(2, len(self)):
            x = self[i](x)
            if i == layer + 7:
                return x
        return x

In [ ]:
# download pretrained model parameters and store them at `param_path`
generator_params_file_id = r"1yT8XomOacOGNZq2CEpvJlxQSeECNhstg"
_ = load_from_url(make_drive_downloadable_url(generator_params_file_id), name="generator.params")
param_path = get_downloaded_file_path(name="generator.params")

# initialise the generator
generator = Generator()
generator.load_parameters(param_path, ctx=ctx)

# PART 1: PGGAN

## Exercise 1: Latent2Face (1 point)
The latent input of PGGAN's generator consists of 512-dimensional hyperspheres where each dimension is independently Gaussian distributed with unit variance.

Your task is to sample a random latent vector, generate a face by feeding it to the generator, and plotting it using matplotlib.

The model expects an input of type [NDArray](https://mxnet.apache.org/versions/1.0.0/api/python/ndarray/ndarray.html) and a layer of type int (choose the last layer=9 if you want the generator to return the face image).

If you are using GPU, don't forget to specify this when converting your input to NDArray, by using `as_in_context`.


In [ ]:
### your code here
latent = np.random.randn(1, 512)

In [ ]:
face = generator(nd.array(latent).as_in_context(ctx), 9).asnumpy()
face = np.clip(np.rint(127.5 * face + 127.5), 0.0, 255.0)
face = face.astype("uint8").transpose(0, 2, 3, 1)

plt.figure()
plt.imshow(face[0])
plt.axis("off")
plt.show()

## Exercise 2: Spherical linear interpolation (1 point)

The GAN latent space has learned smooth transitions that can be visualized by linear interpolation. Because the latent space is a multimodal Gaussian distribution, the "spherical" linear interpolation function $Slerp$ should be used used to take the latents' spherical geometry into account (rather than regular linear interpolation):

\begin{equation}
    Slerp(z_1, z_2, n) = \frac{\sin \big[ (1-n) \Omega \big] }{\sin \Omega} z_1 + \frac{\sin \big[n \Omega \big] }{\sin \Omega} z_2
\end{equation}

where $\Omega$ is the subtended angle which you can compute by taking the dot product of the unit vectors of $z_1$ and $z_2$. That is, $\cos \Omega = \dfrac{z_1 \cdot z_2}{||z_1||*||z_2||}$. Next, $n$ is the interpolation ratio that you can get using <code>np.linspace(0, 1, N)</code> where $N$ is the step size.

First, implement the $Slerp$ function which should return a latent vector. Then, sample two random latents $z_1$ and $z_2$ and use $Slerp$ to obtain N latents which you can feed to the generator to synthesize $N$ face images that transition smoothly from $z_1$ into $z_2$.

<font color= "green">
Answer:


</font>

In [ ]:
def slerp(N, z1, z2):
    """
    Compute the spherical linear interpolation (SLERP) between two vectors in the latent space.

    Parameters:
    N (int): The number of intermediate latent vectors to generate.
    z1 (ndarray): The first latent vector (Numpy array of shape (latent_dim,)).
    z2 (ndarray): The second latent vector (Numpy array of shape (latent_dim,)).

    Returns:
    slerp_latents (ndarray): An array of shape (N, latent_dim) containing the interpolated latent vectors
                             from z1 to z2, including the endpoints.
    """

    z1 = z1 / np.linalg.norm(z1)
    z2 = z2 / np.linalg.norm(z2)

    omega = np.arccos(np.dot(z1, z2))

    latents = []
    for t in np.linspace(0, 1, N):
        latent = (
            np.sin((1 - t) * omega) * z1 +
            np.sin(t * omega) * z2
        ) / np.sin(omega)

        latents.append(latent)

    return np.array(latents)

In [ ]:
N = 8
z1 = np.random.randn(512)
z2 = np.random.randn(512)

interp_latents = slerp(N, z1, z2)

In [ ]:
interp_faces = generator(nd.array(interp_latents, ctx=ctx), 9).asnumpy()
interp_faces = np.clip(np.rint(127.5 * interp_faces + 127.5), 0, 255)
interp_faces = interp_faces.astype("uint8").transpose(0, 2, 3, 1)

plt.figure(figsize=(16, 4))

for i in range(N):
    plt.subplot(1, N, i + 1)
    plt.imshow(interp_faces[i])
    plt.axis("off")

plt.show()

**What does the $Slerp$ function look like in the limit as $\Omega \to 0$?**



Give a short answer below


As $\Omega \to 0$, the two latent vectors point in almost the same direction.

## Exercise 3: Vector arithmetic (1 point)

In the previous exercise, you have seen how image semantics vary with the latent code. The GAN latent space also obeys simple arithmetic operations, such as:

$\texttt{SmilingWoman - NeutralWoman + NeutralMan = SmilingMan}$

Sample and reconstruct a large number of faces. From these faces, select a bunch that correspond to each of the above three categories (i.e., 20 smiling women, 20 neutral women, 20 neutral men). Average the latents per category and perform the arithmetic operations with these averaged latents to obtain a smiling man. If you are feeling creative, feel free to pick any other arithmetic operation instead of the given one.

At the end, you should have a 4 panel figure which shows the faces that go in your operation and the face that comes out.

In [ ]:
num_faces = 20
np.random.seed(42)  # fixed seed so hardcoded indices (smiling_woman_idx etc.) always refer to the same faces
latents = np.random.randn(num_faces, 512).astype(np.float32)

all_faces = []

batch_size = 1

for start in range(0, num_faces, batch_size):
    batch_latents = latents[start:start + batch_size]

    batch_faces = generator(nd.array(batch_latents, ctx=ctx), 10).asnumpy()

    all_faces.append(batch_faces)

faces = np.concatenate(all_faces, axis=0)

faces = np.clip(np.rint(127.5 * faces + 127.5), 0, 255)
faces = faces.astype("uint8").transpose(0, 2, 3, 1)

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(num_faces):
    plt.subplot(4, 5, i + 1)
    plt.imshow(faces[i])
    plt.title(str(i))
    plt.axis("off")

plt.show()

In [ ]:
smiling_woman_idx = [8, 13, 14, 18]
neutral_woman_idx = [4, 5, 12, 17]
neutral_man_idx = [3, 7, 16, 19]

smiling_woman = latents[smiling_woman_idx].mean(axis=0)
neutral_woman = latents[neutral_woman_idx].mean(axis=0)
neutral_man = latents[neutral_man_idx].mean(axis=0)

smiling_man = smiling_woman - neutral_woman + neutral_man

panel_latents = np.stack([
    smiling_woman,
    neutral_woman,
    neutral_man,
    smiling_man
]).astype(np.float32)

panel_faces = generator(nd.array(panel_latents, ctx=ctx), 9).asnumpy()
panel_faces = np.clip(np.rint(127.5 * panel_faces + 127.5), 0, 255)
panel_faces = panel_faces.astype("uint8").transpose(0, 2, 3, 1)

titles = ["Smiling woman", "Neutral woman", "Neutral man", "Result"]

plt.figure(figsize=(12, 4))

for i in range(4):
    plt.subplot(1, 4, i + 1)
    plt.imshow(panel_faces[i])
    plt.title(titles[i])
    plt.axis("off")

plt.show()

## Exercise 4: Semantic face editing (1 point)

In [this paper](https://arxiv.org/pdf/1907.10786.pdf), five decision boundaries are identified for five binary semantic attributes: gender, age, eyeglasses, pose and smile. A face can be edited by editing its underlying latent $z$:

\begin{equation}
    z_{edit} = z + \alpha \textbf{b}
\end{equation}

where $\alpha$'s magnitude and polarity represent the amount of change and direction, respectively, along the separation boundary $\textbf{b}$.

Pick a decision boundary for PGGAN from [their Github](https://github.com/genforce/interfacegan/tree/master/boundaries) and perform semantic face editing on a randomly sampled latent as follows: define a linspace of $n$ $\alpha$'s (E.g., <code>np.linspace(-3, 3, 10)</code> if $n=10$ from $-3$ to $3$) and subtract the dot product between latent and boundary from each element in this linspace. Finally, edit the original latent by adding each $\alpha \textbf{b}$ (as in the equation above) and visualize the $n$ corresponding faces.

We already collected all of the boundary URLs in the block below to make it easier to load and switch once you had a look at the GitHub and decided which one you want to use, or play around with several.

In [ ]:
age_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_age_boundary.npy?raw=true"
age_c_eyeglasses_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_age_c_eyeglasses_boundary.npy?raw=true"
age_c_gender_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_age_c_gender_boundary.npy?raw=true"
age_c_gender_eyeglasses_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_age_c_gender_eyeglasses_boundary.npy?raw=true"
eyeglasses_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_eyeglasses_boundary.npy?raw=true"
eyeglasses_c_age_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_eyeglasses_c_age_boundary.npy?raw=true"
eyeglasses_c_age_gender_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_eyeglasses_c_age_gender_boundary.npy?raw=true"
eyeglasses_c_gender_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_eyeglasses_c_gender_boundary.npy?raw=true"
gender_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_gender_boundary.npy?raw=true"
gender_c_age_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_gender_c_age_boundary.npy?raw=true"
gender_c_age_eyeglasses_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_gender_c_age_eyeglasses_boundary.npy?raw=true"
gender_c_eyeglasses_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_gender_c_eyeglasses_boundary.npy?raw=true"
pose_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_pose_boundary.npy?raw=true"
quality_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_quality_boundary.npy?raw=true"
smile_boundary_url = r"https://github.com/genforce/interfacegan/blob/master/boundaries/pggan_celebahq_smile_boundary.npy?raw=true"

In [ ]:
# load the boundary of your choice
boundary_url = smile_boundary_url # replace with whichever you wish to load
boundary = np.load( io.BytesIO( load_from_url(boundary_url) ) )
### your code here

boundary = boundary.reshape(-1)
boundary = boundary/np.linalg.norm(boundary)

latent = np.random.randn(1, 512).astype(np.float32)

n = 8
alphas = np.linspace(-3, 3, n).astype(np.float32)

edited_latents = []

for alpha in alphas:
    z_edit = latent + alpha * boundary
    edited_latents.append(z_edit[0])

edited_latents = np.array(edited_latents).astype(np.float32)

edited_faces = generator(nd.array(edited_latents, ctx=ctx), 9).asnumpy()
edited_faces = np.clip(np.rint(127.5 * edited_faces + 127.5), 0, 255)
edited_faces = edited_faces.astype("uint8").transpose(0, 2, 3, 1)

plt.figure(figsize=(16, 4))

for i in range(n):
    plt.subplot(1, n, i + 1)
    plt.imshow(edited_faces[i])
    plt.title(f"{alphas[i]:.1f}")
    plt.axis("off")

plt.show()

# PART 2: Dataset


Load in the (latents, responses) dataset. In this experiment, participants were shown faces generated by the model while their brain activity was recorded in an fMRI. This dataset consists of latents that generated the faces participants viewed in the scanner, and the brain activity responses that were recorded as a result of the viewing.

Notice the shapes of the data arrays and become familiar with the data.

In [ ]:
subj1_file_id = r"1GYcje6YNbjzPchQTUJulcIIqDykUOioU"
subj2_file_id = r"1kdi2U4uTMyFmZ3gfbZxb9u4V5PCtDxGX"
subj_url = make_drive_downloadable_url(subj1_file_id)

x_tr, t_tr, x_te, t_te = pickle.load( io.BytesIO( load_from_url(subj_url, name="subject_latents_and_responses.dat") ) )
print((len(x_tr), len(x_te)), (len(t_tr), len(t_te)))

## Exercise 5: Visual stimuli (5 points)

Generate and visualise the faces that correspond to the latents from the test set. These were the visual stimuli in the experiment.

Make sure to save all of the face-images to the respective train and test directory, e.g. by using the provided function

In [ ]:
def save_face(directory, face, face_id):
  Image.fromarray(face, 'RGB').save(directory / f"{face_id}.png")

In [ ]:
t_tr = np.array(t_tr)
t_te = np.array(t_te)
print(t_tr.shape, t_te.shape)

test_dir = __working_directory / "test"
train_dir = __working_directory / "train"

test_dir.mkdir(parents=True, exist_ok=True)
train_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
batch_size = 1
layer = 9

# loading the test faces first because they are faster
for start in tqdm(range(0, len(t_te), batch_size)):
    face_path = test_dir / f"{start}.png"
    if face_path.exists():
        continue
    batch = nd.array(np.array(t_te[start:start+batch_size]).astype(np.float32)).as_in_context(ctx)
    out = generator(batch, layer)
    out_np = out.asnumpy()
    out_np = np.clip(np.rint(127.5 * out_np + 127.5), 0, 255).astype("uint8").transpose(0, 2, 3, 1)
    del out, batch
    mx.nd.waitall()
    save_face(test_dir, out_np[0], start)

In [ ]:
# now populating the training directory
for start in tqdm(range(0, len(t_tr), batch_size)):
    face_path = train_dir / f"{start}.png"
    if face_path.exists():
        continue
    batch = nd.array(np.array(t_tr[start:start+batch_size]).astype(np.float32)).as_in_context(ctx)
    out = generator(batch, layer)
    out_np = out.asnumpy()
    out_np = np.clip(np.rint(127.5 * out_np + 127.5), 0, 255).astype("uint8").transpose(0, 2, 3, 1)
    del out, batch
    mx.nd.waitall()
    save_face(train_dir, out_np[0], start)


## Exercise 6: Brain responses (10 points)

We are going to plot the neural responses on the brain using [Nilearn's glass brain plotting](https://nilearn.github.io/dev/auto_examples/01_plotting/plot_demo_glass_brain.html).

First, take the mean brain response of all the training set responses using np.mean().

Second, use the mask coordinates to bring the 1D (4096) responses back to 3D (79, 95, 79).

Lastly, use the affine transformation matrix from the atlas to align the responses on the glass brain visualization.

In [ ]:
x_tr = np.array(x_tr)
x_te = np.array(x_te)
print("Training set:", x_tr.shape, "Test set:", x_te.shape)

In [ ]:
mask_coords_file_id = r"1aElSThEakHFmuWS9zoSPNfh2-1CySExA"
atlas_mni_file_id = r"1eFUf0sjAdqDNZZsiLPgP_5cvNS5yuqNQ"
mask_coords_url = make_drive_downloadable_url(mask_coords_file_id)
atlas_mni_url = make_drive_downloadable_url(atlas_mni_file_id)

coords = np.load(io.BytesIO( load_from_url(mask_coords_url, name="mask_coordinates.npy") ))
load_from_url(atlas_mni_url, name="glass_atlas.nii")
atlas = nib.Nifti1Image.load(get_downloaded_file_path(name="glass_atlas.nii"))

mu_tr = np.mean(x_tr, axis=0) # this is step 1
atlas.shape, coords.shape, mu_tr.shape

In [ ]:
### your code here
# step 2
brain_vol = np.zeros(atlas.shape[:3])
brain_vol[coords[:, 0], coords[:, 1], coords[:, 2]] = mu_tr

# step 3
brain_img = nib.Nifti1Image(brain_vol, affine=atlas.affine)

# checking that it actually works
plotting.plot_glass_brain(brain_img, colorbar=True, title="Mean training brain response")
plt.show()

## Exercise 7: Glasser brain atlas (5 points)

Load in the [Glasser atlas](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4990127/) and visualize it using glass brain. You can use the plot_glass_brain and plot_roi functions.
<br>
<br>

**Legend**

Early and intermediate visual cortex:<br>
1) Primary_Visual: 1.  
2) Early_Visual: 4, 5, 6.   
3) Dorsal_Stream_Visual: 13, 19, 3, 152, 16, 17.  
4) Ventral_Stream_Visual: 7, 163, 22, 18, 153, 160, 154.  
5) MT+_Complex_and_Neighboring_Visual_Areas: 158, 20, 21, 159, 156, 157, 23, 2, 138.     
   
Sensorimotor areas:<br>
6) Somatosensory_and_Motor: 8, 53, 9, 51, 52.   
7) Paracentral_Lobular_and_Mid_Cingulate: 40, 41, 55, 44, 43, 36, 39, 37.     
8) Premotor: 54, 96, 10, 56, 78, 11.  
9) Posterior_Opercular: 99, 113, 100, 101, 102, 105.      

Auditory regions:<br>
10) Early_Auditory: 24, 174, 173, 124, 104.   
11) Auditory_Association: 175, 125, 129, 128, 130, 176, 123, 107.     
12) Insular_and_Frontal_Opercular: 103, 178, 168, 167, 106, 115, 114, 116, 109, 111, 112, 110, 108, 169.  

Temporal cortex:<br>
13) Medial_Temporal: 120, 119, 118, 122, 126, 155, 127.  
14) Lateral_Temporal: 137, 133, 177, 132, 136, 134, 172, 131, 135.     

Posterior cortex:<br>
15) Temporo-Parieto-Occipital_Junction: 139, 140, 141, 28, 25.   
16) Superior_Parietal: 48, 95, 49, 117, 50, 47, 42, 45, 46, 29.  
17) Inferior_Parietal: 143, 151, 150, 149, 148, 116, 147, 146, 145, 144.  
18) Posterior_Cingulate: 142, 121, 31, 15, 14, 33, 34, 35, 161, 162, 32, 38, 27, 30.     

Anterior cortex:<br>
19) Anterior_Cingulate_and_Medial_Prefrontal: 58, 57, 59, 180, 61, 60, 179, 62, 64, 165, 63, 69, 88, 65, 164.   
20) Orbital_and_Polar_Frontal: 94, 66, 77, 91, 92, 89, 170, 90, 72, 93, 166.   
21) Inferior_Frontal: 74, 75, 80, 79, 81, 82, 76, 171.   
22) Dorsolateral_Prefrontal: 73, 67, 97, 98, 26, 70, 71, 87, 68, 83, 85, 84, 86.*


In [ ]:
from nilearn import datasets

# fetch the atlas
destrieux = datasets.fetch_atlas_destrieux_2009()  # fallback if Glasser unavailable
glasser_url = "https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_1mm.nii.gz"
load_from_url(glasser_url, name="glasser_atlas.nii.gz")
glasser_atlas = nib.load(get_downloaded_file_path(name="glasser_atlas.nii.gz"))
glasser_data = glasser_atlas.get_fdata()

# plot the full Glasser parcellation as glass brain
plotting.plot_glass_brain(glasser_atlas, colorbar=True, title="Glasser Atlas (all regions)", display_mode="lzr")
plt.show()

# Plot with plot_roi to see boundaries
plotting.plot_roi(glasser_atlas, title="Glasser Atlas ROI view",display_mode="ortho",colorbar=True)
plt.show()

region_groups = {
    "Primary_Visual":                       [1],
    "Early_Visual":                         [4, 5, 6],
    "Dorsal_Stream_Visual":                 [13, 19, 3, 152, 16, 17],
    "Ventral_Stream_Visual":                [7, 163, 22, 18, 153, 160, 154],
    "MT+_Complex":                          [158, 20, 21, 159, 156, 157, 23, 2, 138],
    "Somatosensory_and_Motor":              [8, 53, 9, 51, 52],
    "Paracentral_Lobular_Mid_Cingulate":    [40, 41, 55, 44, 43, 36, 39, 37],
    "Premotor":                             [54, 96, 10, 56, 78, 11],
    "Posterior_Opercular":                  [99, 113, 100, 101, 102, 105],
    "Early_Auditory":                       [24, 174, 173, 124, 104],
    "Auditory_Association":                 [175, 125, 129, 128, 130, 176, 123, 107],
    "Insular_Frontal_Opercular":            [103, 178, 168, 167, 106, 115, 114, 116,
                                             109, 111, 112, 110, 108, 169],
    "Medial_Temporal":                      [120, 119, 118, 122, 126, 155, 127],
    "Lateral_Temporal":                     [137, 133, 177, 132, 136, 134, 172, 131, 135],
    "Temporo-Parieto-Occipital_Junction":   [139, 140, 141, 28, 25],
    "Superior_Parietal":                    [48, 95, 49, 117, 50, 47, 42, 45, 46, 29],
    "Inferior_Parietal":                    [143, 151, 150, 149, 148, 116, 147, 146,
                                             145, 144],
    "Posterior_Cingulate":                  [142, 121, 31, 15, 14, 33, 34, 35, 161,
                                             162, 32, 38, 27, 30],
    "Anterior_Cingulate_Medial_Prefrontal": [58, 57, 59, 180, 61, 60, 179, 62, 64,
                                             165, 63, 69, 88, 65, 164],
    "Orbital_Polar_Frontal":                [94, 66, 77, 91, 92, 89, 170, 90, 72, 93, 166],
    "Inferior_Frontal":                     [74, 75, 80, 79, 81, 82, 76, 171],
    "Dorsolateral_Prefrontal":              [73, 67, 97, 98, 26, 70, 71, 87, 68, 83,
                                             85, 84, 86],
}

highlight_groups = ["Primary_Visual",
    "Early_Visual",
    "Ventral_Stream_Visual",
    "Early_Auditory",
    "Somatosensory_and_Motor"]

fig, axes = plt.subplots(len(highlight_groups), 1,
                          figsize=(10, 3 * len(highlight_groups)))

for ax, group_name in zip(axes, highlight_groups):
    label_ids = region_groups[group_name]
    # Build a binary mask volume for this group
    group_vol = np.zeros_like(glasser_data)
    for label_id in label_ids:
        group_vol[glasser_data == label_id] = label_id
    group_img = nib.Nifti1Image(group_vol, affine=glasser_atlas.affine)
    plotting.plot_glass_brain(
        group_img,
        colorbar=False,
        title=group_name.replace("_", " "),
        axes=ax,
        display_mode="lzr")

plt.suptitle("Selected Glasser atlas region groups", fontsize=14)
plt.tight_layout()
plt.show()


# PART 3: Neural encoding

## Exercise 8: Stimulus-feature model (10 points)

[Alexnet](https://papers.nips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf) is one of the first ridiculously successfull computer vision models. Check the authors of the paper and see if you recognise a name 😉.

These days, pretrained model weights are widely avaiable online to avoid training the same models for the same tasks over and over again.
The parameters you will be loading are from an AlexNet that was trained to categorize [ImageNet](https://www.image-net.org) images.

We can extract the models intermediate feature representations to images.

Feed the images from the training and test set to the model and extract the feature representations from multiple layers. Save them as numpy arrays.

In [ ]:
alex_net_params_file_id = r"1iqtJjDbRM9zlYNf-V_ZNnoCV7GGoFAvq"
alex_net_params_url = make_drive_downloadable_url(alex_net_params_file_id)

load_from_url(alex_net_params_url, name="alex_net.params")
alex_net_params_path = get_downloaded_file_path(name="alex_net.params")

In [ ]:
class Alexnet(HybridBlock):
    def __init__(self, layer: int) -> None:
        super(Alexnet, self).__init__()
        self.layer = layer
        with self.name_scope():
            self.features = nn.HybridSequential("")
            with self.features.name_scope():
                if layer >= 1:
                    self.features.add(nn.Conv2D(64, 11, 4, 2, activation="relu"))
                    self.features.add(nn.MaxPool2D(3, 2))
                if layer >= 2:
                    self.features.add(nn.Conv2D(192, 5, padding=2, activation="relu"))
                    self.features.add(nn.MaxPool2D(3, 2))
                if layer >= 3:
                    self.features.add(nn.Conv2D(384, 3, padding=1, activation="relu"))
                if layer >= 4:
                    self.features.add(nn.Conv2D(256, 3, padding=1, activation="relu"))
                if layer >= 5:
                    self.features.add(nn.Conv2D(256, 3, padding=1, activation="relu"))
                    self.features.add(nn.MaxPool2D(3, 2))
                if layer >= 6:
                    self.features.add(nn.Flatten())
                    self.features.add(nn.Dense(4096, activation="relu"))
                    self.features.add(nn.Dropout(0.5))
                if layer >= 7:
                    self.features.add(nn.Dense(4096, activation="relu"))
                    self.features.add(nn.Dropout(0.5))
            if layer == 8:
                self.output = nn.Dense(1000)
        self.load_parameters(alex_net_params_path, ctx, ignore_extra=True)

    def transform(self, image):
        resized = mx.image.resize_short(image, 224)
        cropped, crop_info = mx.image.center_crop(resized, (224, 224))
        normalized = mx.image.color_normalize(cropped.astype(np.float32)/255,
                                        mean=mx.nd.array([0.485, 0.456, 0.406]),
                                        std=mx.nd.array([0.229, 0.224, 0.225]))
        transposed = normalized.transpose((2, 0, 1))
        batchified = transposed.expand_dims(axis=0)
        return batchified.as_in_context(ctx)

    def hybrid_forward(self, F: ModuleType, x: nd.NDArray) -> nd.NDArray:
        x = self.transform(x)
        return self.output(self.features(x)) if self.layer == 8 else self.features(x)


In [ ]:
# Example: feed stimulus 0.png to alexnet and extract the feature representation from layer 1.
# this example assumes you saved your test images into the test directory and
# used a simple count as file name, so we are loading the image "0.png"
# if you used a different naming scheme, change the code to reflect that
layers = [1]
for layer in layers:
    feature_model = Alexnet(layer)
    stimulus = mx.image.imread(test_dir / "0.png")
    feature = feature_model(stimulus).asnumpy()
    print(stimulus.shape, feature.shape)

In [ ]:
feature_dir = __working_directory / "features"
feature_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
### your code here
layers = [1,2,3,4,5,6,7,8]
for layer in layers:
    feature_model = Alexnet(layer)
    features_train = []
    features_test = []

    # for training
    for item in range(len(t_tr)):
        stimulus = mx.image.imread(train_dir / f"{item}.png")
        feature = feature_model(stimulus).asnumpy()
        feature = feature.flatten() #flatten the feature
        features_train.append(feature)

    #for testing
    for item in range(len(t_te)):
        stimulus = mx.image.imread(test_dir / f"{item}.png")
        feature = feature_model(stimulus).asnumpy()
        feature = feature.flatten()
        features_test.append(feature)

    # converting to npy arrays
    features_train = np.array(features_train) # train features
    features_test = np.array(features_test) # test features

    # saving as .npy
    np.save(file = feature_dir/f"train_layer_{layer}", arr = features_train)
    np.save(file = feature_dir/f"test_layer_{layer}", arr = features_test)

## Exercise 9: Feature-response model (10 points)

Fit a linear feature-response model on the extracted Alexnet features and responses from the training set using [Scikit's RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html) (play around with different alpha values). Specifically, you are asked to use the features from the previous exercise to predict the brain activity given in the dataset.

You could train different models for the voxels of different brain regions according to the Glasser atlas, but the main goal is to understand how to implement encoding models from feature representations. Train $k$ different models on features from $k$ different Alexnet layers.

Evaluate the predicted responses versus the real responses from the test set using pearson_correlation_coefficient() per voxel (i.e., each voxel has its own correlation coefficient). In the end, you should have $k$ correlation coefficients per voxel response, and assign the layer that resulted in the highest correlation coefficient to each voxel.

Plot the your results (e.g., assigned layers or correlation coefficients, depending on what you're interested in) on the glass brain visualization.

In [ ]:
def pearson_correlation_coefficient(x: np.ndarray, y: np.ndarray, axis: int) -> np.ndarray:
    r = (np.nan_to_num(zscore(x)) * np.nan_to_num(zscore(y))).mean(axis)
    p = 2 * t.sf(np.abs(r / np.sqrt((1 - r ** 2) / (x.shape[0] - 2))), x.shape[0] - 2)
    return r, p

In [ ]:
import gc
from sklearn.linear_model import Ridge

layers    = [1, 2, 3, 4, 5, 6, 7, 8]
n_voxels  = x_tr.shape[1]
all_r     = np.zeros((len(layers), n_voxels), dtype=np.float32)

# pre computed average test responses
n_repeats = len(x_te) // len(t_te)
x_te_avg  = x_te.reshape(n_repeats, len(t_te), n_voxels).mean(axis=0)  # (n_te, n_voxels)

# number of voxels
CHUNK = 256

for i, layer in enumerate(layers):
    print(f"\n Layer {layer}")

    X_tr = np.load(feature_dir / f"train_layer_{layer}.npy",mmap_mode="r")
    X_te = np.load(feature_dir / f"test_layer_{layer}.npy",mmap_mode="r")
    feat_dim = X_tr.shape[1]
    print(f" Feature dimension: {feat_dim}")

    # pick alpha
    sample_vox = min(64, n_voxels)

    rcv = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], store_cv_results=False)
    rcv.fit(np.asarray(X_tr), x_tr[:, :sample_vox])
    best_alpha = float(rcv.alpha_)
    print(f"Best alpha: {best_alpha}")

    del rcv
    gc.collect()

    # fit the voxels in chunks
    y_pred_all = np.zeros((len(t_te), n_voxels), dtype=np.float32)

    for v_start in range(0, n_voxels, CHUNK):
        v_end = min(v_start + CHUNK, n_voxels)
        ridge = Ridge(alpha=best_alpha)
        ridge.fit(np.asarray(X_tr), x_tr[:, v_start:v_end])

        y_pred_all[:, v_start:v_end] = (ridge.predict(np.asarray(X_te)).astype(np.float32))

        del ridge
        gc.collect()

    r, p = pearson_correlation_coefficient(x_te_avg,y_pred_all,axis=0)
    all_r[i] = r.astype(np.float32)
    print(f"Layer {layer} mean r: {r.mean():.4f}")
    del X_tr, X_te, y_pred_all
    gc.collect()

best_layer = np.argmax(all_r, axis=0) + 1
print("\nBest layer distribution:", np.bincount(best_layer))

# best layer
best_layer_vol = np.zeros(atlas.shape[:3])
best_layer_vol[coords[:, 0], coords[:, 1], coords[:, 2]] = best_layer
best_layer_img = nib.Nifti1Image(best_layer_vol, affine=atlas.affine)
plotting.plot_glass_brain(best_layer_img, colorbar=True,
                           title="Best Alexnet layer per voxel")
plt.show()

# max correlation
max_r_vol = np.zeros(atlas.shape[:3])
max_r_vol[coords[:, 0], coords[:, 1], coords[:, 2]] = all_r.max(axis=0)
max_r_img = nib.Nifti1Image(max_r_vol, affine=atlas.affine)
plotting.plot_glass_brain(max_r_img, colorbar=True,
                           title="Max correlation per voxel")
plt.show()


# PART 4: Neural decoding

## Exercise 10: response-latent model (10 points)

Fit a linear response-feature decoding model that predicts GAN latents from brain responses. How does your model performance change as a function of regularization? Make sure to vary your regularization parameter and show how reconstruction performance changes as a function of it.



In [ ]:
### your code here
x_tr_arr = np.array(x_tr)
x_te_arr = np.array(x_te)
t_tr_arr = np.array(t_tr)
t_te_arr = np.array(t_te)
# repeated test measures
n_repeats = len(x_te_arr) // len(t_te_arr)
x_te_avg = x_te_arr.reshape(n_repeats, len(t_te_arr), -1).mean(axis=0)
print("Averaged test responses shape:", x_te_avg.shape)  # (36, 4096)
# different regularization values
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
scores = []
for alpha in alphas:
    ridge = RidgeCV(alphas=[alpha])
    ridge.fit(x_tr_arr, t_tr_arr)
    latents_pred = ridge.predict(x_te_avg)
    r, _ = pearson_correlation_coefficient(t_te_arr, latents_pred, axis=0)
    scores.append(r.mean())
    print(f"alpha={alpha}: mean r={r.mean():.4f}")
# plot
plt.figure()
plt.semilogx(alphas, scores, marker='o')
plt.xlabel("Alpha (regularization)")
plt.ylabel("Mean Pearson r")
plt.title("Decoding performance vs regularization")
plt.grid(True)
plt.show()
# best alpha
best_alpha = alphas[np.argmax(scores)]
print(f"Best alpha: {best_alpha}")
final_decoder = RidgeCV(alphas=[best_alpha])
final_decoder.fit(x_tr_arr, t_tr_arr)
latents_pred_te = final_decoder.predict(x_te_avg)
print("Predicted latents shape:", latents_pred_te.shape)  # (36, 512)

## Exercise 11: reconstruction (10 points)

Feed your predicted latents to the generator and report your reconstruction performance (e.g., feature similarity or pearson's correlation coefficient over pixels). Show the faces generated by the target latents and the faces generated by your predicted latents.

In [ ]:
### your code here

# generate faces from latents
recon_faces = []
for i in tqdm(range(len(latents_pred_te))):
    batch = nd.array(latents_pred_te[i:i+1].astype(np.float32)).as_in_context(ctx)
    out = generator(batch, 9)
    out_np = out.asnumpy()
    del out, batch
    mx.nd.waitall()
    out_np = np.clip(np.rint(127.5 * out_np + 127.5), 0, 255).astype("uint8").transpose(0, 2, 3, 1)
    recon_faces.append(out_np[0])
recon_faces = np.array(recon_faces)
print("Reconstructed faces shape:", recon_faces.shape)

# loading targeted test faces
target_faces = []
for i in range(len(t_te)):
    face = np.array(Image.open(test_dir / f"{i}.png"))
    target_faces.append(face)
target_faces = np.array(target_faces)
# pixel wise pearson
target_flat = target_faces.reshape(len(target_faces), -1).astype(float)
recon_flat = recon_faces.reshape(len(recon_faces), -1).astype(float)
r, _ = pearson_correlation_coefficient(target_flat, recon_flat, axis=1)
print(f"Mean pixel wise Pearson r: {r.mean()}")
print(f"Per image r: {r}")
# target vs reconstructed together for 6
n_show = 6
plt.figure(figsize=(18, 6))
for i in range(n_show):
    plt.subplot(2, n_show, i + 1)
    plt.imshow(target_faces[i])
    plt.title(f"Target {i}")
    plt.axis("off")

    plt.subplot(2, n_show, n_show + i + 1)
    plt.imshow(recon_faces[i])
    plt.title(f"r={r[i]:.2f}")
    plt.axis("off")
plt.suptitle("Target vs Reconstructed faces")
plt.tight_layout()
plt.show()

# PART 5: Your experiment

## Exercise 12: report (20 points)

Now that you have mastered the basics, you can perform your own experiment with this dataset of brain responses to synthesized stimuli. Write an "ultra-short" (800-1,600 words) report on your experiment where you state your research question, methods, results and discussion.

### Food for thought / research ideas:

*   Improve reconstructions with a custom loss function. In addition to mean squared error between predicted and real latent, also add other loss terms (e.g., structural similarity index, Alexnet/VGGFace features)

*   Linear interpolation of fMRI responses
*   Vector arithmetic with fMRI responses
*   Neural encoding using VGGFace features for face recognition
*   Neural encoding: predict custum face attributes (e.g., emotional expression) by averaging many latents (as in exercise 3) or training your own SVM (as in exercise 4).
*   Neural encoding / decoding from voxels in specific brain areas according to the Glasser atlas










# Which visual brain regions are most useful for face reconstruction?

## Research question

The goal of this experiment was to test whether different visual brain regions contribute equally to brain-to-face reconstruction, or whether some visual regions carry more useful information than others. In exercises 10 and 11 the model was trained from brain responses to GAN latent vectors, then the predicted latents were fed into the generator to reconstruct faces. In this experiment, we changed the question from "How well can the full model reconstruct faces?" to "Which visual brain regions provide the strongest reconstruction signal?"

The main question of this experiment was:

**Do early visual cortex or ventral visual cortex voxels produce better GAN-latent predictions?**

We expected that both visual regions would contain useful information, but that ventral visual cortex might perform slightly better because the ventral visual stream is more strongly associated with object and face-related visual processing. Early visual cortex may preserve lower-level visual features, while ventral visual cortex may preserve more complex face-related structure.

## Methods

We used the same fMRI response data, PGGAN latent vectors, and generated face stimuli as in the previous reconstruction exercises. For the region comparison, we used two visual voxel groups only. The first group was an early visual cortex group, combining primary and early visual cortex labels. The second group was a ventral visual cortex group, combining ventral visual atlas labels associated with higher-level visual processing. These groups were selected using the Glasser atlas labels assigned to the available fMRI voxels. The early visual cortex group contained 426 voxels, while the ventral visual cortex group contained 504 voxels. To avoid giving one group an advantage simply because it contained more input features, both groups were reduced to the same number of voxels. Therefore, 426 voxels were used from each group.

For each group, we trained a Ridge regression decoder to predict the 512-dimensional PGGAN latent vector from the selected fMRI voxels. The regularization parameter was kept fixed using the value selected in the previous exercises. Repeated test responses were averaged before evaluation.

The main evaluation metrics were latent mean squared error and mean latent Pearson correlation. Image reconstruction was not used as the main metric in this simplified experiment, because passing predicted latents through the generator for several region models was computationally expensive and we did not have enough GPU power. Instead, we focused on whether the selected voxel groups could predict the GAN latent representation.

## Results

| region                | n_voxels | latent_mse | latent_corr |
| :-------------------- | -------: | ---------: | ----------: |
| Ventral visual cortex |      426 |     1.1936 |      0.0196 |
| Early visual cortex   |      426 |     1.1637 |      0.0180 |

The ventral visual cortex group achieved a slightly higher mean latent correlation than the early visual cortex group. This suggests that ventral visual areas may contain slightly more useful information for predicting the GAN latent representation of faces. This fits the expectation that ventral visual cortex is important for higher-level object and face-related visual processing.

However, the early visual cortex group achieved a slightly lower latent MSE. Therefore, the result is mixed: ventral visual cortex performed better according to latent correlation, while early visual cortex performed better according to latent MSE. The difference between the two groups was also small, so the results should be taken with a grain of salt.

Overall, both early and ventral visual regions produced weak but measurable latent prediction performance. The result does not show a strong advantage for one visual region over the other, but it suggests that ventral visual cortex may have a small advantage when performance is measured using latent correlation.

## Discussion

This experiment extends the earlier decoding exercises by asking where the useful reconstruction signal comes from. The original idea was to compare visual cortex with non-visual control regions. However, the selected non-visual control labels contained only 6 available voxels, which made that comparison too underpowered. To create a more meaningful and balanced experiment, we changed the final comparison to early visual cortex versus ventral visual cortex, which allowed both groups to use 426 voxels.

The results were mixed, with ventral visual cortex having a slightly higher latent correlation, which may indicate that it captured more face-relevant structure in the GAN latent space. This is consistent with the role of the ventral stream in object and face-related processing. On the other hand, early visual cortex had a slightly lower latent MSE, suggesting that it may have predicted the overall latent values slightly more accurately under this metric.

Because the differences were small, the result are not very conclusive. The latent correlations were still close to zero, meaning that neither region produced strong latent predictions on its own. This may be because the number of test examples is small or the relevant information is distributed across multiple visual and non-visual areas instead of being isolated in one region.

Another limitation is that only latent-space metrics were used in the final lightweight version of the experiment. Image reconstruction metrics such as pixel correlation and image MSE were not computed, because generating reconstructed faces for several region models was too computationally taxing. Latent correlation and latent MSE still give useful information about decoder performance, but they do not directly measure perceptual similarity between reconstructed and target faces.

Overall, this experiment suggests that both early and ventral visual regions contain some information useful for GAN-latent prediction. Ventral visual cortex showed a small advantage in latent correlation, while early visual cortex showed a small advantage in MSE.


In [ ]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

x_tr_arr = np.asarray(x_tr)
x_te_arr = np.asarray(x_te)
t_tr_arr = np.asarray(t_tr)
t_te_arr = np.asarray(t_te)

# avg repeated test responses
n_repeats = len(x_te_arr) // len(t_te_arr)
x_te_avg = x_te_arr.reshape(n_repeats, len(t_te_arr), -1).mean(axis=0)

print("Train responses:", x_tr_arr.shape)
print("Averaged test responses:", x_te_avg.shape)
print("Train latents:", t_tr_arr.shape)
print("Test latents:", t_te_arr.shape)

atlas_data = atlas.get_fdata()
voxel_labels = atlas_data[coords[:, 0], coords[:, 1], coords[:, 2]].astype(int)

region_labels = {
    "Early visual cortex": [1, 4, 5, 6],
    "Ventral visual cortex": [7, 163, 22, 18, 153, 160, 154],
}

region_alpha = float(best_alpha)

MAX_VOXELS_PER_GROUP = 600 # for gpu's sake
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


def mean_latent_correlation(y_true, y_pred):
    # mean pearson correlation across the 512 latent dims
    r, _ = pearson_correlation_coefficient(y_true, y_pred, axis=0)
    return float(np.nanmean(r))


available_voxel_counts = []

for region_name, labels in region_labels.items():
    mask = np.isin(voxel_labels, labels)
    voxel_idx = np.where(mask)[0]
    available_voxel_counts.append(len(voxel_idx))
    print(region_name, "available voxels:", len(voxel_idx))

N_VOXELS_PER_GROUP = min(MAX_VOXELS_PER_GROUP, min(available_voxel_counts))

print("\nUsing", N_VOXELS_PER_GROUP, "voxels per group")


region_results = []
region_predictions = {}

for region_name, labels in region_labels.items():
    mask = np.isin(voxel_labels, labels)
    voxel_idx = np.where(mask)[0]

    n_voxels_available = len(voxel_idx)

    if n_voxels_available == 0:
        print(f"Skipping {region_name}: no voxels found in mask")
        continue

    n_voxels = N_VOXELS_PER_GROUP

    voxel_idx = rng.choice(voxel_idx, size=n_voxels, replace=False)

    print(f"\nTraining decoder for {region_name} ({n_voxels} voxels)")

    decoder = Ridge(alpha=region_alpha, solver="lsqr")
    decoder.fit(x_tr_arr[:, voxel_idx], t_tr_arr)

    pred_latents = decoder.predict(x_te_avg[:, voxel_idx])

    latent_mse = mean_squared_error(t_te_arr, pred_latents)
    latent_r = mean_latent_correlation(t_te_arr, pred_latents)

    region_results.append({
        "region": region_name,
        "n_voxels": n_voxels,
        "latent_mse": latent_mse,
        "latent_corr": latent_r,
    })

    region_predictions[region_name] = pred_latents


region_df = pd.DataFrame(region_results).sort_values("latent_corr", ascending=False)

print("\nRegion decoding results:")
display(region_df)

plt.figure(figsize=(6, 4))
plt.bar(region_df["region"], region_df["latent_corr"])
plt.ylabel("Mean latent correlation")
plt.title("Early Visual vs Ventral Visual latent prediction")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(region_df["region"], region_df["latent_mse"])
plt.ylabel("Latent MSE")
plt.title("Early Visual vs Ventral Visual latent prediction error")
plt.tight_layout()
plt.show()